![image_1779966968785.png](./image_1779966968785.png "image_1779966968785.png")

![image_1779966987588.png](./image_1779966987588.png "image_1779966987588.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date
from pyspark.sql import functions as f
# Initialize Spark session
spark = SparkSession.builder.appName("TradesUsersDataFrame").getOrCreate()

# ================================
# Create trades DataFrame
# ================================
trades_data = [
    (10, 201, 15, "Completed", "2023-07-01"),
    (11, 202, 20, "Completed", "2023-07-05"),
    (12, 203, 10, "Completed", "2023-07-10"),
    (13, 204, 8, "Completed", "2023-07-15"),
    (14, 205, 25, "Completed", "2023-07-20"),
    (15, 201, 30, "Completed", "2023-07-25"),
    (16, 202, 5, "Cancelled", "2023-08-01"),
    (17, 203, 12, "Completed", "2023-08-05"),
    (18, 206, 18, "Completed", "2023-08-10"),
    (19, 206, 22, "Completed", "2023-08-15"),
    (20, 207, 9, "Cancelled", "2023-08-20"),
    (21, 204, 14, "Completed", "2023-08-25"),
    (22, 205, 7, "Completed", "2023-09-01")
]

trades_columns = ["order_id", "user_id", "quantity", "status", "trade_date"]

df_trades = spark.createDataFrame(trades_data, trades_columns) \
    .withColumn("trade_date", to_date("trade_date", "yyyy-MM-dd"))

# ================================
# Create users DataFrame
# ================================
users_data = [
    (201, "Boston", "user1@email.com"),
    (202, "Boston", "user2@email.com"),
    (203, "Seattle", "user3@email.com"),
    (204, "Austin", "user4@email.com"),
    (205, "Seattle", "user5@email.com"),
    (206, "Denver", "user6@email.com"),
    (207, "Austin", "user7@email.com")
]

users_columns = ["user_id", "city", "email"]

df_users = spark.createDataFrame(users_data, users_columns)

# ================================
# Show DataFrames
# ================================
df_trades.show()
df_users.show()

In [0]:
result_df = (
    df_trades.join(df_users, df_trades.user_id == df_users.user_id)
    .groupBy(df_users.city)
    .agg(
        f.sum(f.when(df_trades.status == "Completed", 1).otherwise(0)).alias(
            "total_quantity"
        ),
    )
    .select("city", "total_quantity")
    .orderBy("total_quantity", ascending=False)
    .limit(3)
)
display(result_df)